# Clase 7 — Gráficos interactivos con Plotly

Hasta ahora tus gráficos eran imágenes estáticas: los generas, los miras, y ahí se quedan. Si quieres saber el valor exacto de un punto, tienes que adivinarlo mirando el eje. Si el gráfico tiene muchos datos amontonados, no puedes acercarte para verlos mejor.

En la vida real —y en cualquier reporte o dashboard que entregues a otra persona— eso no basta. Quien mira el gráfico quiere pasar el mouse sobre un punto y ver su valor exacto, hacer zoom en una zona concreta, o esconder una serie con un clic. Eso es exactamente lo que te da **Plotly**.

**Analogía:** pasar de matplotlib/seaborn a Plotly es como pasar de una foto impresa a una foto en tu celular. La foto impresa la miras y ya está. La foto del celular la puedes tocar, hacer zoom con los dedos, deslizar. El contenido puede ser el mismo, pero la experiencia de explorarlo cambia por completo.

## Dos formas de usar Plotly

Plotly en Python tiene dos caras, y conviene que sepas cuál usar y cuándo:

- **`plotly.express`** (se importa como `px`): interfaz de alto nivel. Con una sola línea de código creas un gráfico completo, con título y ejes incluidos. Es el punto de partida recomendado cuando empiezas: menos código, menos fricción.
- **`plotly.graph_objects`** (se importa como `go`): interfaz de bajo nivel. Construyes el gráfico pieza por pieza —primero una figura vacía, luego le añades "trazas" (`traces`), luego ajustas el diseño (`layout`)—. Te da más control, pero requiere más líneas.

Por dentro, `plotly.express` en realidad usa `plotly.graph_objects`: es un envoltorio (wrapper) que te ahorra trabajo para los casos más comunes.

Vas a ver ambas en este capítulo, usando el dataset de aerolíneas (`airline_data.csv`, con información de vuelos domésticos en EE. UU.) que ya conoces de capítulos anteriores.

`plotly` viene preinstalado en Google Colab, así que no hace falta instalarlo con `pip install`.

## 0. Preparación: librerías y datos

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Cargamos el dataset de aerolíneas
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str}
)

print('Datos cargados correctamente:', airline_data.shape)
airline_data.head()

In [ ]:
# El dataset completo es enorme (cientos de miles de filas), así que tomamos
# una muestra aleatoria de 500 filas para trabajar más rápido.
# random_state=42 asegura que siempre obtengas la misma muestra.
data = airline_data.sample(n=500, random_state=42)

data.shape

## 1. Scatter plot con `graph_objects`

Un scatter (dispersión) muestra la relación entre dos variables. Aquí usamos la interfaz de bajo nivel para ver cómo se construye un gráfico paso a paso: figura vacía → `add_trace()` → `update_layout()`.

In [ ]:
# Paso 1: creamos una figura vacía
fig = go.Figure()

# Paso 2: le añadimos una "traza" de tipo scatter con add_trace().
# mode='markers' dibuja puntos (en vez de líneas).
fig.add_trace(go.Scatter(
    x=data['Distance'],
    y=data['DepTime'],
    mode='markers',
    marker=dict(color='red')
))

# Paso 3: actualizamos el título y las etiquetas de los ejes
fig.update_layout(
    title='Distance vs Departure Time',
    xaxis_title='Distance',
    yaxis_title='DepTime'
)

fig.show()

Con `fig.show()` obtienes un gráfico donde puedes pasar el mouse sobre cada punto y ver sus valores exactos, o hacer zoom arrastrando el mouse sobre una zona.

**Nota:** un line plot se construye con el mismo `go.Scatter()`, cambiando únicamente `mode='markers'` por `mode='lines'`.

## 2. Bar chart con `plotly.express`

Cambiamos a la interfaz de alto nivel. Con `px`, defines todo —datos, ejes y título— en una sola llamada.

In [ ]:
# Agrupamos por estado de destino y sumamos el total de vuelos
bar_data = data.groupby(['DestState'])['Flights'].sum().reset_index()

# Una sola línea: pasamos el dataframe, x, y y el título
fig = px.bar(bar_data, x="DestState", y="Flights",
             title='Total number of flights to the destination state split by reporting airline')
fig.show()

Compara esto con el scatter de más arriba: mismo resultado interactivo, muchísimo menos código.

## 3. Histograma con `plotly.express`

Un histograma agrupa datos continuos en rangos (bins) y muestra cuántos valores caen en cada rango. Útil para ver la forma de una distribución.

In [ ]:
# Reemplazamos valores faltantes por 0 antes de graficar
data['ArrDelay'] = data['ArrDelay'].fillna(0)

fig = px.histogram(data, x="ArrDelay",
                    title="Total number of flights to the destination state split by reporting air.")
fig.show()

## 4. Bubble chart: el scatter con una tercera dimensión

Un bubble chart es un scatter plot al que le añades una tercera variable: el tamaño de cada punto (la "burbuja"). Así, en un solo gráfico comunicas tres cosas a la vez en vez de dos.

Nota que un bubble chart en Plotly Express **no tiene su propia función**: se hace con `px.scatter()` y el parámetro `size`.

In [ ]:
# Agrupamos por aerolínea y sumamos el número de vuelos
bub_data = data.groupby('Reporting_Airline')['Flights'].sum().reset_index()

# size="Flights" hace que el tamaño de cada burbuja represente el número de vuelos.
# hover_name define qué texto aparece en negrita al pasar el mouse.
fig = px.scatter(bub_data, x="Reporting_Airline", y="Flights", size="Flights",
                  hover_name="Reporting_Airline",
                  title='Reporting Airline vs Number of Flights', size_max=60)
fig.show()

## 5. Pie chart

Un pie chart (gráfico circular) muestra qué proporción representa cada categoría dentro de un total.

In [ ]:
# values define el tamaño de cada porción, names define su etiqueta
fig = px.pie(data, values='Month', names='DistanceGroup',
             title='Distance group proportion by month')
fig.show()

## 6. Sunburst: cuando una jerarquía no cabe en un pie chart

Todos los gráficos anteriores comparan categorías en un solo nivel. Pero ¿qué pasa si tus datos tienen niveles anidados, como mes → estado de destino → número de vuelos? Ahí es donde el pie chart se queda corto y entra el **sunburst**.

Un sunburst representa datos jerárquicos con círculos concéntricos: el círculo más interno es la raíz (el nivel más general), y cada anillo hacia afuera representa un nivel más específico de la jerarquía. Es como un pie chart, pero con "capas" hacia afuera que puedes explorar haciendo clic para hacer zoom en una rama concreta.

In [ ]:
# path define el orden de la jerarquía: primero Month (círculo interno),
# luego DestStateName (anillo externo). values define el tamaño de cada sector.
fig = px.sunburst(data, path=['Month', 'DestStateName'], values='Flights',
                   title='Flight Distribution Hierarchy')
fig.show()

Al pasar el mouse por el círculo interno ves el total de vuelos de ese mes; al mirar el anillo externo ves cómo se reparte ese total entre los distintos estados de destino. Eso —mostrar un total y su desglose en un mismo gráfico interactivo— es lo que un bar chart o un pie chart normales no te dan de forma directa.

## Puntos clave

- Plotly convierte gráficos estáticos en gráficos interactivos: puedes hacer zoom, pasar el mouse para ver valores exactos, y explorar los datos en vez de solo mirarlos.
- `plotly.express` (`px`) es la interfaz de alto nivel: menos código, buen punto de partida.
- `plotly.graph_objects` (`go`) es la interfaz de bajo nivel: figura vacía → `add_trace()` → `update_layout()`. Más control, más líneas.
- Scatter y line plot en `graph_objects` usan la misma función (`go.Scatter`); solo cambia el parámetro `mode` (`'markers'` vs `'lines'`).
- Un bubble chart es un scatter con una tercera variable codificada en el tamaño (`size=`).
- Un sunburst muestra jerarquías (varios niveles anidados) con círculos concéntricos — algo que un pie chart de un solo nivel no puede representar.

## Siguiente paso

Abre `clase07_practica_plotly-interactivo.ipynb` y resuelve los ejercicios propuestos.